# Indexes and Query Performance

Imagine you have a table with 10 million rows, and you write a query to find one specific customer: `SELECT * FROM Customers WHERE email = 'alice@email.com'`. 

By default, the database doesn't know where Alice is. So, it starts at row 1 and checks every single row all the way to row 10,000,000 until it finds her. This is called a **Full Table Scan**, and it is the enemy of database performance.

To fix this, we use **Indexes**. An index is a special data structure (usually a B-Tree) that the database builds behind the scenes. It keeps a sorted list of your column's data and acts as a shortcut map, allowing the database to find your row instantly without scanning the whole table.

Let's use Python and SQLite to see exactly how the database "thinks" when searching for data.

In [1]:
import sqlite3
import pandas as pd
import random

# 1. Connect to an in-memory database
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# 2. Create a Users table
cursor.executescript("""
CREATE TABLE Users (
    user_id INTEGER PRIMARY KEY,
    username TEXT,
    department TEXT
);
""")

# 3. Let's simulate a large database by inserting 100,000 dummy rows
print("Generating 100,000 rows of data. Please wait a moment...")
departments = ['Engineering', 'Sales', 'HR', 'Marketing', 'Finance']
dummy_data = []

for i in range(1, 100001):
    # We will specifically make one unique user we want to search for later
    if i == 75000:
        dummy_data.append((i, 'TargetUser99', 'Data Science'))
    else:
        dummy_data.append((i, f'User{i}', random.choice(departments)))

cursor.executemany("INSERT INTO Users (user_id, username, department) VALUES (?, ?, ?)", dummy_data)
print("✅ Database ready with 100,000 users!")

Generating 100,000 rows of data. Please wait a moment...
✅ Database ready with 100,000 users!


# 1. The Problem: Full Table Scans
Before we actually run a query, we can ask the database *how* it plans to execute it using the `EXPLAIN QUERY PLAN` command. This is a Data Scientist's best friend for debugging slow SQL code.

Let's ask SQLite how it plans to find our specific user.

In [2]:
# We ask the database for its execution plan
bad_plan_query = "EXPLAIN QUERY PLAN SELECT * FROM Users WHERE username = 'TargetUser99';"

print("--- Query Plan WITHOUT an Index ---")
display(pd.read_sql_query(bad_plan_query, conn))

--- Query Plan WITHOUT an Index ---


,id,parent,notused,detail
0,2,0,216,SCAN Users


*(Notice the output says **SCAN TABLE**. "Scan" is the dangerous word here. It means the database has to read every single row in the table until it finds 'TargetUser99'.)*

# 2. Creating an Index
Creating an index is simple. You just tell the database which table and which column you want to optimize. 

Because we know we will be searching for users by their `username` frequently, we should put an index on that column.

In [3]:
# Create an index on the username column
create_index_sql = "CREATE INDEX idx_username ON Users (username);"
cursor.execute(create_index_sql)

print("✅ Index 'idx_username' successfully created!")

✅ Index 'idx_username' successfully created!


# 3. The Solution: Index Searches (Seeks)
Now that the index is built, let's ask the database for its execution plan again using the exact same query.

In [4]:
# Ask for the execution plan again
good_plan_query = "EXPLAIN QUERY PLAN SELECT * FROM Users WHERE username = 'TargetUser99';"

print("--- Query Plan WITH an Index ---")
display(pd.read_sql_query(good_plan_query, conn))

--- Query Plan WITH an Index ---


,id,parent,notused,detail
0,3,0,62,SEARCH Users USING INDEX idx_username (usernam...


*(Notice the output now says **SEARCH TABLE** using the index. "Search" (or "Seek" in some databases) means it used the shortcut map. Instead of checking 100,000 rows, it found the user in just 2 or 3 steps!)*

# 4. Multi-Column (Composite) Indexes
If your queries frequently filter by multiple columns together (e.g., `WHERE department = 'Engineering' AND username = 'Alice'`), you can create an index that covers both columns. 

In [5]:
# Create an index on both department and username
cursor.execute("CREATE INDEX idx_dept_user ON Users (department, username);")
print("✅ Composite index created!")

✅ Composite index created!


*(Note: Order matters in composite indexes! If you index `(department, username)`, it helps queries filtering by `department` alone, or both. It does **not** help queries filtering by `username` alone. Always put the column you filter by most often first!)*

# 5. The Dark Side of Indexing (Trade-offs)
If indexes make `SELECT` queries incredibly fast, why don't we just put an index on every single column in the database?

1. **Storage Space**: Indexes are actual data structures saved on the hard drive. If you index every column, your database size could double or triple.
2. **Slower Writes (`INSERT`, `UPDATE`, `DELETE`)**: Every time you add a new user to the table, the database has to update the main table AND stop to update the index. If a table has 15 indexes, one simple `INSERT` statement has to write data to 16 different places.

**The Golden Rule:** Only index the columns you frequently use in your `WHERE`, `JOIN`, and `ORDER BY` clauses.

In [6]:
# Clean up
conn.close()

## Real-World Use Case or Analogy:
Think of a Database Table like a massive **1,000-page Textbook**:

* **Full Table Scan (No Index)**: Your professor asks you to find every page that mentions "Machine Learning". The book has no index at the back. You have to start at page 1 and read every single word on every single page until you reach page 1,000. It takes you hours.
* **Creating the Index (`CREATE INDEX`)**: The publisher hires an editor. The editor reads the book once, finds all the important keywords, sorts them alphabetically, and prints a 10-page "Index" at the back of the book. (This takes extra time and adds 10 extra pages to the book's physical size).
* **Index Search**: Your professor asks you to find "Machine Learning". You flip straight to the 'M' section of the index at the back of the book. It says "Machine Learning: Pages 42, 105, 800". You flip directly to those three pages. You found your data in 10 seconds instead of 10 hours.

---